In [ ]:
import pandas as pd
import seaborn as sns
from utils import parse_har_files_to_dataframe
from common import BASE_HAR_DIR


sns.set_theme(style="whitegrid", context="paper")
pd.set_option('display.max_colwidth', 128)

df = parse_har_files_to_dataframe(BASE_HAR_DIR, test=False)

In [ ]:
GTM_JS_URL_PREFIX = 'https://www.googletagmanager.com/gtm.js?id=GTM-'

gtm_js_scripts_by_source = set(df[df.has_gtm_signature].url.unique())
gtm_js_scripts_by_url = set(df[df.url.str.startswith(GTM_JS_URL_PREFIX)].url.unique())
gtm_js_urls = gtm_js_scripts_by_source.union(gtm_js_scripts_by_url)

# requests that are triggered by GTM
df['triggered_by_gtm'] = df.initiator_urls.apply(
    lambda x: any([url in gtm_js_urls for url in x]))


df['has_gtm_param'] = df.param_names.apply(lambda x: 'gtm' in x)

# intersection of all GA4 parameters that are triggered by GTM
ga4_params_set = df[
    df.triggered_by_gtm &
    df.has_gtm_param & (df.entity != 'Google')
    ]['param_names'].map(set)

from functools import reduce
ga4_intersection = reduce(set.intersection, ga4_params_set)

# label requests that contain all of the params of the intersection
df['has_intersect_ga4_params'] = df['param_names'].apply(lambda x: ga4_intersection.issubset(x))
# Display the intersection
print(len(ga4_intersection), sorted(list(ga4_intersection)))

36 ['_p', '_s', 'are', 'cid', 'dl', 'dma', 'dma_cps', 'dt', 'ecid', 'frm', 'gcd', 'gcs', 'gtm', 'npa', 'pscdl', 'sct', 'seg', 'sid', 'sr', 'sst.rnd', 'sst.tft', 'sst.ude', 'tag_exp', 'tfd', 'tid', 'uaa', 'uab', 'uafvl', 'uam', 'uamb', 'uap', 'uapv', 'uaw', 'ul', 'ur', 'v']


In [4]:
df['triggered_by_gtm'].value_counts()

triggered_by_gtm
False    229852
True      12271
Name: count, dtype: int64

In [5]:
df[df.has_gtm_signature & (df.entity != "Google")].url.value_counts()

url
https://tmsst.aponeo.de/gtm.js?id=GTM-5GNVZ2M              4
https://sgtm.mycare.de/gtm.js?id=GTM-NR6HHR                4
https://measure.docmorris.de/gtm.js?id=GTM-KBGX7K8         4
https://pipeline.drogist.nl/webevents.js?id=GTM-TVJBDC     4
https://sgtm.plein.nl/load?id=GTM-5N8W39                   4
https://sst.koopjesdrogisterij.nl/gtm.js?id=GTM-N45WVXP    4
https://sgtm.docpeter.it/mapwbued.js?id=GTM-KV9B4PQ        4
https://sgtm.farmasave.it/wyxyzwjg.js?id=GTM-NP6XD6D       3
Name: count, dtype: int64

In [6]:
df.req_hostname.nunique()

900

In [7]:
df[df.url.str.contains('trustedshops.com') & df.is_tracker]

,site_hostname,exp_mode,country,consent_mode,url,initiator_type,initiator_urls,param_names,post_mime_type,post_data,...,cname_entity,entity,site_domain,third_party,initiator_domains_str,is_tracker,is_cname_tracker,triggered_by_gtm,has_gtm_param,has_intersect_ga4_params
2858,www.pharmashopi.com,social,FR,Accept,https://widgets.trustedshops.com/reviews/tsSticker/tsSticker.js,script,[https://www.pharmashopi.com/],[],,,...,,Trusted Shops,pharmashopi.com,True,pharmashopi.com,True,False,False,False,False
2924,www.pharmashopi.com,social,FR,Accept,https://widgets.trustedshops.com/js/XBF10FC8790D633F938DCE5C0DF880247.js,script,[https://www.pharmashopi.com/],[],,,...,,Trusted Shops,pharmashopi.com,True,pharmashopi.com,True,False,False,False,False
2958,www.pharmashopi.com,social,FR,Accept,https://widgets.trustedshops.com/reviews/tsSticker/css/light.theme.css,script,[https://widgets.trustedshops.com/reviews/tsSticker/tsSticker.js],[],,,...,,Trusted Shops,pharmashopi.com,True,trustedshops.com,True,False,False,False,False
2959,www.pharmashopi.com,social,FR,Accept,https://widgets.trustedshops.com/assets/trustbadge.js,script,[https://widgets.trustedshops.com/js/XBF10FC8790D633F938DCE5C0DF880247.js],[],,,...,,Trusted Shops,pharmashopi.com,True,trustedshops.com,True,False,False,False,False
2963,www.pharmashopi.com,social,FR,Accept,https://widgets.trustedshops.com/assets/fr-FR.b5295866478355662c4b.chunk.js,script,"[https://widgets.trustedshops.com/assets/trustbadge.js, https://widgets.trustedshops.com/js/XBF10FC8790D633F938DCE5C0DF88024...",[],,,...,,Trusted Shops,pharmashopi.com,True,trustedshops.com,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241962,www.farmaciaevacontreras.com,social,ES,Reject,https://widgets.trustedshops.com/assets/images/sprite.fa10bbbffbb158ef65643d1dccd20ba7bd355392.svg,script,"[https://widgets.trustedshops.com/js/XFC4E91F429D9347839039CB5CE4FA227.js?ver=2.0.3, https://widgets.trustedshops.com/assets...",[],,,...,,Trusted Shops,farmaciaevacontreras.com,True,trustedshops.com,True,False,False,False,False
242048,www.farmaciaevacontreras.com,social,ES,Reject,https://widgets.trustedshops.com/js/XFC4E91F429D9347839039CB5CE4FA227.js?ver=2.0.3,,[],[ver],,,...,,Trusted Shops,farmaciaevacontreras.com,True,,True,False,False,False,False
242106,www.farmaciaevacontreras.com,social,ES,Reject,https://widgets.trustedshops.com/assets/trustbadge.js,script,[https://widgets.trustedshops.com/js/XFC4E91F429D9347839039CB5CE4FA227.js?ver=2.0.3],[],,,...,,Trusted Shops,farmaciaevacontreras.com,True,trustedshops.com,True,False,False,False,False
242108,www.farmaciaevacontreras.com,social,ES,Reject,https://widgets.trustedshops.com/assets/es-ES.ad5caad35bf8b7e0210e.chunk.js,script,"[https://widgets.trustedshops.com/js/XFC4E91F429D9347839039CB5CE4FA227.js?ver=2.0.3, https://widgets.trustedshops.com/assets...",[],,,...,,Trusted Shops,farmaciaevacontreras.com,True,trustedshops.com,True,False,False,False,False


In [8]:
import socket
import whois
import os
import json
from functools import cache

WHOIS_CACHE_FILE = 'whois_cache.json'

@cache
def get_whois_info(host):
    ip_address = socket.gethostbyname(host)
    whois_info = whois.whois(ip_address)
    whois_org = whois_info.get('org', '') if whois_info.get('org') else ''
    whois_domains = whois_info.get('domain_name')
    if isinstance(whois_domains, list):
        whois_domains = list(set([domain.upper() for domain in whois_domains]))
        whois_domains = ' '.join(whois_domains) if whois_domains else ''
    elif whois_domains:
        whois_domains = whois_domains.upper()
    else:
        whois_domains = ''
    return whois_org.upper() + '\t' + whois_domains


def load_whois_cache(cache_file, hostnames):
    whois_dict = {}
    if os.path.exists(cache_file):
        print(f'Loading whois cache from {cache_file}')
        with open(cache_file, 'r') as f:
            return json.load(f)

    # Update the whois_dict with new entries
    for req_hostname in hostnames:
        if req_hostname not in whois_dict:
            whois_dict[req_hostname] = get_whois_info(req_hostname)
            # print(req_hostname, get_whois_info(req_hostname))
    # Save the updated cache
    with open(WHOIS_CACHE_FILE, 'w') as f:
        json.dump(whois_dict, f)

    return whois_dict



In [9]:
whois_dict = load_whois_cache(WHOIS_CACHE_FILE, df[
    df.has_intersect_ga4_params].req_hostname.unique())

df['whois_info'] = df.req_hostname.apply(lambda x: whois_dict.get(x, ''))

Loading whois cache from whois_cache.json


In [10]:
GOOGLE_SERVERS = [
    'Google Tag Manager',
    'Google Frontend'
    ]

GOOGLE_WHOIS = [
    "GOOGLE LLC-GOOGLEUSERCONTENT.COM",
    "GOOGLE LLC-1E100.NET",
    "-GOOGLEUSERCONTENT.COM"
    ]


def is_hosted_by_google(row):
    """Not used for detection"""
    whois = row['whois_info']
    if ((whois in GOOGLE_WHOIS) or
        ('1.1 google' in row['via']) or
            (row['server'].strip() in GOOGLE_SERVERS)):
        return True
    return False

###########################################################
df['google_hosted'] = df.apply(is_hosted_by_google, axis=1)
###########################################################


In [11]:
# ###########################################################
# gtm_js_scripts_by_source = set(df[df.has_gtm_signature].url.unique())
# gtm_js_scripts_by_url = set(df[df.url.str.startswith(GTM_JS_URL_PREFIX)].url.unique())
# gtag_js_scripts = set(df[df.url.str.startswith(GTAG_JS_URL_PREFIX)].url.unique())
# # all requests that are triggered by GTM or gtag
# gtm_gtag_triggered = gtm_js_scripts_by_source.union(gtag_js_scripts).union(gtm_js_scripts_by_url)

# # gtm_gtag_triggered = gtm_js_scripts_by_source.union(gtm_js_scripts_by_url)
# # label requests that are triggered by GTM or gtag
# df['triggered_by_gtm'] = df.initiator_urls.apply(lambda x: any([url in gtm_gtag_triggered for url in x]))
# ###########################################################

# SST = Server Side Tagging
# label requests that are triggered by (GTM or gtag) and contain all GA4 parameters
df['sst'] = df.has_intersect_ga4_params
# df['sst'] = df.has_intersect_ga4_params & df.triggered_by_gtm

df['url_path'] = df.url.apply(lambda x: x.split('?')[0])
df['first_party'] = ~df.third_party

dft = df[df.sst]
df.sst.value_counts()

sst
False    240565
True       1558
Name: count, dtype: int64

In [12]:
dft.site_domain.nunique(), dft.site_domain.value_counts()

(19,
 site_domain
 plein.nl                 226
 docmorris.de             130
 trekpleister.nl          128
 etos.nl                  107
 kruidvat.nl              107
 koopjesdrogisterij.nl     92
 deonlinedrogist.nl        88
 farmasave.it              86
 drogist.nl                85
 medpex.de                 78
 redcare.it                75
 farmaciasdirect.es        71
 da.nl                     62
 shop-apotheke.com         54
 docpeter.it               44
 aponeo.de                 43
 efarma.com                34
 mycare.de                 26
 soin-et-nature.com        22
 Name: count, dtype: int64)

In [13]:
dft.entity.value_counts()

entity
                                                           1141
DocMorris                                                   130
Ahold Delhaize                                              107
Comventure                                                   78
Shop-Apotheke                                                54
myCARE                                                       26
PHARMACIE MR COMAILLS ARNAUD ET MELLE COMAILLS DELPHINE      22
Name: count, dtype: int64

In [14]:
dft.groupby(['exp_mode', 'consent_mode', 'country'])['site_domain'].\
    nunique().to_frame()


site_domain
exp_mode consent_mode country             
social   Accept       DE                 5
                      ES                 1
                      IT                 4
                      NL                 8
         Reject       DE                 1
                      ES                 1
                      FR                 1
                      IT                 4
                      NL                 8
tracking Accept       DE                 5
                      ES                 1
                      IT                 4
                      NL                 8
         Reject       DE                 1
                      ES                 1
                      FR                 1
                      IT                 4
                      NL                 8

In [15]:
COLS = ['site_domain', 'req_hostname', 'country']


In [16]:
dft.drop_duplicates(COLS).req_hostname.nunique(),\
    dft[dft.exp_mode!='tracking'].req_hostname.nunique(),\
        dft[dft.consent_mode=='Reject'].req_hostname.nunique(),\
            dft[dft.initiator_domains_str.str.contains('googletagmanager.com')].req_hostname.nunique()



(19, 19, 15, 15)

In [17]:
# dft = dft[dft.exp_mode!='tracking']
dft[dft.exp_mode=='tracking'].drop_duplicates(['consent_mode', 'site_domain', 'req_hostname'])[
    ['consent_mode', 'country', 'site_domain', 'req_hostname']].\
        sort_values(['consent_mode', 'country'])

,consent_mode,country,site_domain,req_hostname
67317,Accept,DE,medpex.de,measure.medpex.de
68741,Accept,DE,aponeo.de,tmsst.aponeo.de
71702,Accept,DE,shop-apotheke.com,klpoz.shop-apotheke.com
75413,Accept,DE,mycare.de,sgtm.mycare.de
81021,Accept,DE,docmorris.de,measure.docmorris.de
121173,Accept,ES,farmaciasdirect.es,datos.farmaciasdirect.es
83913,Accept,IT,redcare.it,otasf.redcare.it
92041,Accept,IT,efarma.com,gtm.efarma.com
97688,Accept,IT,farmasave.it,sgtm.farmasave.it
99785,Accept,IT,docpeter.it,sgtm.docpeter.it


In [18]:
# no of rows where param names include sst.etld
# no of rows where param names include sst.etld
dft.shape[0], dft[dft.param_names.apply(lambda x: 'sst.etld' in x)].shape[0]

(1558, 1310)

In [25]:
df[df.param_names.apply(lambda x: 'email' in x)][['req_domain', 'site_domain', 'url']]

,req_domain,site_domain,url
8295,pharma-gdd.com,pharma-gdd.com,https://www.pharma-gdd.com/fr/ajax/login/test-email?email=green.monika24@gmail.com
8297,pharma-gdd.com,pharma-gdd.com,https://www.pharma-gdd.com/fr/register?email=green.monika24@gmail.com
38369,squeezely.tech,drogist.nl,https://t.squeezely.tech/t?e=PageView&email=greenemily5676%40gmail.com&newtracker=true&log_id=5850721d-b86f-48cc-9a5c-d5614b...
39173,deonlinedrogist.nl,deonlinedrogist.nl,https://activate.deonlinedrogist.nl/t?e=CompleteRegistration&custom_archived=no&email=greenemily5676%40gmail.com&custom_cust...
39174,deonlinedrogist.nl,deonlinedrogist.nl,https://activate.deonlinedrogist.nl/t?e=EmailOptIn&newsletter=yes&service=yes&custom_review_consent=yes&custom_origin_source...
39175,deonlinedrogist.nl,deonlinedrogist.nl,https://activate.deonlinedrogist.nl/t?e=PrePurchase&language=nl-NL&email=greenemily5676%40gmail.com&firstname=emily&lastname...
89735,1000farmacie.it,1000farmacie.it,https://backend.1000farmacie.it/api/v1/account/user_lookup?email=green.monika24%40gmail.com
89736,1000farmacie.it,1000farmacie.it,https://backend.1000farmacie.it/api/v1/account/user_lookup?email=green.monika24%40gmail.com
108932,pharma-gdd.com,pharma-gdd.com,https://www.pharma-gdd.com/fr/ajax/login/test-email?email=green.monika24@gmail.com
108939,pharma-gdd.com,pharma-gdd.com,https://www.pharma-gdd.com/fr/register?email=green.monika24@gmail.com


In [21]:
df[(df.exp_mode == 'tracking') & df.sst].drop_duplicates(COLS)[
    ['country', 'site_domain', 'req_hostname',
    'google_hosted', 'via', 'server', 'whois_info', 'cname']]

,country,site_domain,req_hostname,google_hosted,via,server,whois_info,cname
67317,DE,medpex.de,measure.medpex.de,True,1.1 google,Google Frontend,GOOGLE LLC\tGOOGLEUSERCONTENT.COM,
68741,DE,aponeo.de,tmsst.aponeo.de,True,1.1 google,Google Frontend,GOOGLE LLC\tGOOGLEUSERCONTENT.COM,
71702,DE,shop-apotheke.com,klpoz.shop-apotheke.com,True,1.1 google,Google Frontend,GOOGLE LLC\tGOOGLEUSERCONTENT.COM,
75413,DE,mycare.de,sgtm.mycare.de,False,,,\t,
81021,DE,docmorris.de,measure.docmorris.de,True,1.1 google,Google Frontend,GOOGLE LLC\tGOOGLEUSERCONTENT.COM,
83913,IT,redcare.it,otasf.redcare.it,True,1.1 google,Google Frontend,GOOGLE LLC\tGOOGLEUSERCONTENT.COM,
92041,IT,efarma.com,gtm.efarma.com,False,,Apache/2.4.62 (Debian),\tGROWEB.IT,ihcjdeaiapzd.eu-1.tagmanagerserverside.it
97688,IT,farmasave.it,sgtm.farmasave.it,False,,,GOOGLE LLC\tGOOGLEUSERCONTENT.COM,
99785,IT,docpeter.it,sgtm.docpeter.it,False,,,GOOGLE LLC\tGOOGLEUSERCONTENT.COM,
121173,ES,farmaciasdirect.es,datos.farmaciasdirect.es,True,1.1 google,,GOOGLE LLC\t1E100.NET,ghs.googlehosted.com


In [22]:
reject_hosts = set(df[df.sst & (df.consent_mode == 'Reject')].req_hostname.unique())
accept_hosts = set(df[df.sst & (df.consent_mode == 'Accept')].req_hostname.unique())
a_r_hosts = accept_hosts.intersection(reject_hosts)
only_accept_hosts = accept_hosts - a_r_hosts
only_reject_hosts = reject_hosts - a_r_hosts
print('Accept hosts:', len(accept_hosts), 'Reject hosts:', len(reject_hosts), 'Accept-Reject hosts:', len(a_r_hosts))
print('Only Accept hosts:', only_accept_hosts, '\nOnly Reject hosts:', only_reject_hosts)
df['gtm_consent'] = df.req_hostname.apply(lambda x: 'Only in accept' if x in only_accept_hosts else 'Only in reject' if x in only_reject_hosts else 'Both' if x in a_r_hosts else 'No GTM consent')

Accept hosts: 18 Reject hosts: 15 Accept-Reject hosts: 14
Only Accept hosts: {'sgtm.mycare.de', 'measure.docmorris.de', 'measure.medpex.de', 'tmsst.aponeo.de'} 
Only Reject hosts: {'care.soin-et-nature.com'}


In [ ]:
print(df[(df.exp_mode == 'tracking') & df.sst].drop_duplicates(COLS)[
    ['country', 'url_path', 'google_hosted', 'gtm_consent']
].to_latex(index=False, header=['Country', 'SST Hostname', 'Google Hosted', 'Consent']))

In [31]:
unique_req_hostnames = df[(df.exp_mode == 'tracking') & df.sst].req_hostname.unique()
pd.DataFrame(unique_req_hostnames, columns=['req_hostname']).to_csv('detected_sst_req_hostnames.csv', index=False)